# Custom AI Receipt Parser - Training Pipeline
This notebook is designed to train the **Donut (Document Understanding Transformer)** model
to read receipts and map them directly to JSON format.

**IMPORTANT:** Training a vision model usually requires a GPU. Since you are training on a CPU with 16GB RAM,
the settings below have been heavily optimized to reduce memory footprint so it doesn't crash your computer.

## 1. Install Dependencies
Run this cell to install everything required for training.

In [ ]:
!pip install -q transformers datasets torch torchvision "accelerate>=1.1.0" pillow

In [ ]:
import os
import re
import torch
from datasets import load_dataset
from PIL import Image
from transformers import (
    VisionEncoderDecoderConfig,
    VisionEncoderDecoderModel,
    DonutProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

## 2. Load the Dataset
We will use the Hugging Face `naver-clova-ix/cord-v2` dataset (Consolidated Receipt Dataset).
This dataset contains thousands of receipts with bounding boxes and parsed JSON strings.

In [ ]:
print("Loading CORD dataset...")
# Load a very small subset for CPU training to avoid Out-Of-Memory errors
dataset = load_dataset("naver-clova-ix/cord-v2", split="train[:100]") 

print(f"Loaded {len(dataset)} examples.")
# Let's see what a sample looks like
sample = dataset[0]
print("Sample Image Size:", sample["image"].size)
print("Sample ground truth:", sample["ground_truth"])

## 3. Initialize the Donut Processor and Model
Donut treats image parsing as a translation task: "Image -> text sequence".

In [ ]:
model_id = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(model_id)
model = VisionEncoderDecoderModel.from_pretrained(model_id)

# Enable Gradient Checkpointing to save massive amounts of RAM during training
model.config.gradient_checkpointing = True

# Set image size and tokens
# Reduced image size slightly to save RAM on CPU
image_size = [960, 720] 
model.config.encoder.image_size = image_size
model.config.decoder.max_length = 512

# Add special tokens for receipt structuring
new_tokens = ["<s_receipt>", "</s_receipt>", "<s_total>", "</s_total>", "<s_items>", "</s_items>", "<s_item>", "</s_item>"]
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_tokens + [processor.tokenizer.eos_token]})
model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(["<s_receipt>"])[0]

print("Model and Processor loaded successfully.")

## 4. Create the PyTorch Dataset for Training
We must format the raw images into PyTorch Tensors and convert the JSON strings into token IDs.

In [ ]:
from torch.utils.data import Dataset

class ReceiptDataset(Dataset):
    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = example["image"].convert("RGB")
        
        # Format the target text sequence. In real fine-tuning, you map your own JSON structure here.
        # This dataset natively returns a complex JSON string. Let's append our start/end tokens.
        target_sequence = "<s_receipt>" + example["ground_truth"] + processor.tokenizer.eos_token
        
        # Process the image into pixel values
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        
        # Process the text into token IDs
        labels = self.processor.tokenizer(
            target_sequence,
            add_special_tokens=False,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )["input_ids"].squeeze()

        # We tell the model to ignore padding tokens during loss calculation
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

train_dataset = ReceiptDataset(dataset, processor)
print("PyTorch Dataset ready.")

## 5. Configure Training Arguments & Start Training
We use Hugging Face's `Seq2SeqTrainer` to handle the training loop.
These settings are hyper-optimized so that it fits within 16GB of System RAM on a CPU.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./receipt_parser_model",
    per_device_train_batch_size=1, # Lowest possible batch size to save RAM
    gradient_accumulation_steps=8, # Simulates a batch size of 8 while only using RAM for 1
    num_train_epochs=1,            # 1 epoch for testing on CPU (CPU training is very slow)
    learning_rate=2e-5,
    logging_steps=5,
    save_steps=50,
    fp16=False,                    # CPU does not natively support FP16 well, turning off
    use_cpu=True,                  # Forcing CPU usage
    dataloader_num_workers=0,      # Setting to 0 prevents Python from spawning RAM-hungry duplicate processes
    gradient_checkpointing=True,   # Major memory saver, trades compute time to save RAM
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting CPU training! Warning: CPU training for vision models is extremely slow.")
# trainer.train()  # UNCOMMENT THIS LINE TO START TRAINING

## 6. Save the Model
Once training finishes, save the weights so you can load them in an API later.

In [ ]:
# UNCOMMENT THE LINES BELOW AFTER TRAINING
# trainer.save_model("./receipt_parser_model/final_weights")
# processor.save_pretrained("./receipt_parser_model/final_weights")
# print("Model saved locally! You can now load it into a FastAPI server for UniGuard Wallet.")

## 7. Test the Model (Inference)
Once your model is trained, or if you just want to test the pre-trained Donut base model, you can run this block.
Upload a real receipt image to the same folder as this notebook and rename it to `my_receipt.jpg`.

In [ ]:
def test_receipt(image_path="ss.jpeg"):
    if not os.path.exists(image_path):
        print(f"[{image_path}] not found! Please make sure this image exists in your folder.")
        return

    print(f"Running visual inference on: {image_path}")
    image = Image.open(image_path).convert("RGB")
    
    test_model_id = "./receipt_parser_model/final_weights"
    
    if not os.path.exists(test_model_id):
        print(f"Could not find trained model at {test_model_id}. Make sure you ran the 'Save Model' block first!")
        print("Defaulting back to base Donut model for inference...")
        test_model_id = "naver-clova-ix/donut-base"
        
    test_processor = DonutProcessor.from_pretrained(test_model_id)
    test_model = VisionEncoderDecoderModel.from_pretrained(test_model_id)
    
    # Ensuring CPU inference
    device = "cpu"
    test_model.to(device)
    test_model.eval()

    # Process Image
    pixel_values = test_processor(image, return_tensors="pt").pixel_values
    pixel_values = pixel_values.to(device)

    # Generate Sequence
    decoder_input_ids = test_processor.tokenizer(
        "<s_receipt>", 
        add_special_tokens=False, 
        return_tensors="pt"
    ).input_ids
    
    outputs = test_model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids.to(device),
        max_length=test_model.config.decoder.max_length,
        early_stopping=True,
        pad_token_id=test_processor.tokenizer.pad_token_id,
        eos_token_id=test_processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=1,
        bad_words_ids=[[test_processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )

    # Decode Output
    sequence = test_processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(test_processor.tokenizer.eos_token, "").replace(test_processor.tokenizer.pad_token, "")
    sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()  # remove first task start token
    
    print("\n--- AI Model JSON Extraction ---")
    
    # Let the processor turn the Donut string syntax into a real Python dictionary
    parsed_json = test_processor.token2json(sequence)
    print(parsed_json)

# Run the test
# test_receipt("ss.jpeg")